# 00 — Data Acquisition

**Fish Habitat / Potential Fishing Zone (PFZ) prediction — Problem Statement B**

This notebook populates the **raw zone**: it pulls every external dataset the
habitat model needs and caches it on disk. Nothing here is modelling; the goal
is a reproducible, traceable set of inputs.

| Source | What it provides | Role |
|---|---|---|
| Copernicus Marine — physics | SST, salinity, currents, sea-surface height, mixed-layer depth | environmental covariates |
| Copernicus Marine — biogeochemistry | chlorophyll, nitrate, phosphate, silicate, O₂, primary production | prey-base covariates |
| NOAA ERDDAP (`etopo180`) | bathymetry / seafloor relief | static habitat structure |
| OBIS | species occurrence records | **labels** (presences) |
| OBIS (Actinopterygii) | all ray-finned fish records | **background pool** for pseudo-absences |

Everything runs through `marine_ml.sources`, the shared ingestion layer, so the
HAB pipeline and this one fetch data identically.

---

## Two decisions that shape everything downstream

**1. The date window is set by the labels, not by what Copernicus can serve.**
OBIS occurrence records for these species in this region are concentrated in
**2000–2013** and effectively stop after 2014. Training on more recent
environmental fields would mean fields with no labels attached to them. We
verify this empirically at the end of the notebook rather than taking it on
trust.

**2. The region has to be wide.** A small Arabian Sea box returns single-digit
record counts for most target species. The northern Indian Ocean box
(55–95°E, 5°S–25°N) returns 280–400 for the tunas. Occurrence density, not
convenience, picks the box.

**Cadence is monthly.** Occurrence records carry imprecise dates — often just a
month or a year — so daily environmental fields would be false precision
against this label source. Monthly means are also ~30× cheaper to fetch.

In [ ]:
# Run from the `machine_learning` directory, or adjust the path below.
import sys, time, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from marine_ml import config, viz
from marine_ml.sources import copernicus, gebco, obis

viz.use_house_style()
config.ensure_directories()

print(f"project root : {PROJECT_ROOT}")
print(f"raw zone     : {config.RAW_DIR}")

## 1. Region and window

These come from `marine_ml/config.py` rather than being redefined here, so the
notebook and the production pipeline can never drift apart.

In [ ]:
REGION = config.NORTH_INDIAN_OCEAN
START = config.HABITAT_START
END = config.HABITAT_END

print(f"region : {REGION.name}")
print(f"  lon  : {REGION.west}°E → {REGION.east}°E")
print(f"  lat  : {REGION.south}°N → {REGION.north}°N")
print(f"window : {START} → {END}  ({(END - START).days / 365.25:.1f} years)")
print(f"grid   : {config.GRID_RESOLUTION}° common analysis grid")
print()
print("target species:")
for key, name in config.TARGET_SPECIES.items():
    print(f"  {key:18} {name}")

## 2. Copernicus Marine — physical ocean state

Two things about this fetch are load-bearing and were established by timing real
requests, not by reading documentation:

- **`arco-time-series`, never `arco-geo-series`.** Copernicus stores the same
  data two ways. `arco-geo-series` puts one timestep in each huge lat/lon chunk
  — right for "whole globe, latest snapshot", catastrophic here, because it
  would fetch the globe once per timestep in our range.
- **The depth bound must be sent to the server.** These reanalysis products
  carry 50 depth levels and we use only the surface. With
  `minimum_depth`/`maximum_depth` a request takes ~60 s; without it, the
  identical request did not finish in 15 minutes.

Both are handled inside `copernicus.fetch_physics`. Results are cached as
NetCDF, so re-running this cell is free.

In [ ]:
t0 = time.time()
physics = copernicus.fetch_physics(REGION, START, END, cadence="monthly")
print(f"fetched in {time.time() - t0:.1f}s")

print(f"\ndimensions : {dict(physics.sizes)}")
print(f"variables  : {sorted(str(v) for v in physics.data_vars)}")
print(f"time range : {str(physics.time.values[0])[:10]} → {str(physics.time.values[-1])[:10]}")
physics

In [ ]:
# Sanity check: are the values physically plausible, and how much is ocean?
rows = []
for name in physics.data_vars:
    a = physics[name]
    valid = float(a.notnull().mean())
    rows.append({
        "variable": str(name),
        "valid_fraction": round(valid, 3),
        "min": round(float(a.min()), 3) if valid else np.nan,
        "median": round(float(a.median()), 3) if valid else np.nan,
        "max": round(float(a.max()), 3) if valid else np.nan,
        "units": a.attrs.get("units", ""),
    })
pd.DataFrame(rows)

The ~20% of cells that are NaN are land. Sea-surface temperature in the high
20s °C and salinity near 35 PSU are what the northern Indian Ocean should look
like — if either were wildly off, the problem would be the fetch, not the ocean.

## 3. Copernicus Marine — biogeochemistry

Chlorophyll is the prey-base proxy: phytoplankton concentration drives the food
chain that fish aggregate along. Nutrients (nitrate, phosphate, silicate)
explain *why* chlorophyll is where it is.

Note this is **model-based** biogeochemistry rather than satellite ocean colour.
That is deliberate: it is not cloud-limited, so coverage is continuous. Satellite
ocean colour is higher fidelity where it exists but has large seasonal gaps
under monsoon cloud — precisely when the interesting blooms happen.

In [ ]:
t0 = time.time()
bgc = copernicus.fetch_bgc(REGION, START, END, cadence="monthly")
print(f"fetched in {time.time() - t0:.1f}s")

print(f"\ndimensions : {dict(bgc.sizes)}")
print(f"variables  : {sorted(str(v) for v in bgc.data_vars)}")
print(f"\nnote the coarser grid than physics — 1/4° vs 1/12°.")
print("The fusion layer regrids both onto one common 1/4° grid so a feature")
print("means the same thing regardless of which product it came from.")
bgc

## 4. Bathymetry

Depth and seafloor slope are static habitat structure. Shelf breaks and steep
slopes concentrate fish reliably enough that they are among the strongest
predictors in most habitat models.

Served from NOAA ERDDAP's `etopo180` — the same global relief grid GEBCO
publishes, reachable over plain HTTP with no account. ERDDAP returns *altitude*
(positive up, ocean negative); `fetch_bathymetry` converts it to depth
(positive down, land NaN) once, here, rather than in every consumer.

In [ ]:
t0 = time.time()
bathymetry = gebco.fetch_bathymetry(REGION)
print(f"fetched in {time.time() - t0:.1f}s")
print(f"dimensions : {dict(bathymetry.sizes)}")
print(f"depth range: {float(bathymetry.depth.min()):.0f} – {float(bathymetry.depth.max()):.0f} m")
print(f"ocean cells: {float(bathymetry.depth.notnull().mean()):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Sequential ramp: one hue, light→dark, because depth is pure magnitude.
# Coarsened purely for render speed — the analysis uses the full grid.
depth_plot = bathymetry.depth.coarsen(latitude=4, longitude=4, boundary="trim").mean()
mesh = ax.pcolormesh(
    depth_plot.longitude, depth_plot.latitude, depth_plot.values,
    cmap=viz.SEQUENTIAL, shading="auto",
)
viz.annotate_land(ax, bathymetry, REGION)
cbar = fig.colorbar(mesh, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label("depth (m)", color=viz.INK_SECONDARY)
cbar.outline.set_visible(False)

viz.label_axes(
    ax,
    title="Bathymetry of the study region",
    subtitle="Shelf (light) vs abyssal plain (dark). The narrow shelf off India's west coast is where most fishing happens.",
    xlabel="longitude (°E)", ylabel="latitude (°N)",
)
plt.tight_layout()
plt.show()

## 5. OBIS — species occurrence records (the labels)

This is where the labels come from, and it is worth being precise about what
they are.

**OBIS is presence-only.** A record says "this species was observed here on this
date". There is no record anywhere in it saying "this species was looked for
here and was not found". So the negative class does not exist and has to be
constructed — that is notebook `02`'s job, and it is the single most
consequential decision in the whole problem.

We fetch two things:

1. **Target-species presences** → the positive class.
2. **The whole target group** (Actinopterygii, all ray-finned fishes) → *not*
   absences, but the pool from which background points will be drawn. Points
   where somebody demonstrably sampled and recorded *some* fish carry the same
   survey-effort bias as our presences, which is what makes the bias cancel.

In [ ]:
presence_frames = []
for key, scientific_name in tqdm(config.TARGET_SPECIES.items(), desc="OBIS species"):
    frame = obis.fetch_occurrences(scientific_name, REGION, START, END)
    presence_frames.append(frame.assign(species_key=key))

presences = pd.concat(presence_frames, ignore_index=True)
print(f"\n{len(presences)} presence records across {presences.species_key.nunique()} species")
presences.head()

In [ ]:
counts = (
    presences.groupby("species_key")
    .agg(records=("latitude", "size"),
         first_year=("observation_date", lambda s: s.min().year),
         last_year=("observation_date", lambda s: s.max().year),
         datasets=("dataset_id", "nunique"))
    .sort_values("records", ascending=False)
)
counts

`datasets` is worth watching. A species whose records come from one or two
source datasets is one survey programme's itinerary, not an unbiased picture of
where the animal lives — and a model can learn that itinerary. We return to this
in the sampling-bias diagnostic in notebook `01`.

In [ ]:
t0 = time.time()
target_group = obis.fetch_target_group(REGION, START, END)
print(f"fetched in {time.time() - t0:.1f}s")
print(f"{len(target_group)} Actinopterygii records — the background pool")
print(f"spanning {target_group.observation_date.min().date()} → {target_group.observation_date.max().date()}")
print(f"\nratio of background pool to presences: {len(target_group) / len(presences):.1f}×")

## 6. Verifying the window choice

The claim was that occurrence records stop after ~2014, and that this — not
Copernicus coverage — is what bounds the training window. Rather than assert it,
here is the evidence.

In [ ]:
# Deliberately query WITHOUT the date filter so we can see the true distribution.
full_history = []
for key, scientific_name in tqdm(config.TARGET_SPECIES.items(), desc="full history"):
    frame = obis.fetch_occurrences(
        scientific_name, REGION,
        start=pd.Timestamp("1950-01-01").date(),
        end=pd.Timestamp("2024-12-31").date(),
    )
    full_history.append(frame.assign(species_key=key))

full_history = pd.concat(full_history, ignore_index=True)
by_year = full_history.assign(year=full_history.observation_date.dt.year)
yearly = by_year[by_year.year >= 1990].groupby("year").size()
print(f"{len(full_history)} records total, {len(by_year[by_year.year >= 1990])} since 1990")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Single series → one colour, with emphasis on the chosen window rather than
# a value-ramp (bar height already encodes the count).
in_window = [(START.year <= y <= END.year) for y in yearly.index]
ax.bar(yearly.index, yearly.values,
       color=[viz.PRESENCE if inside else viz.INK_MUTED for inside in in_window],
       width=0.75)

ax.axvspan(START.year - 0.5, END.year + 0.5, color=viz.PRESENCE, alpha=0.06, zorder=0)
ax.annotate("training window", xy=((START.year + END.year) / 2, ax.get_ylim()[1] * 0.92),
            ha="center", color=viz.PRESENCE, fontsize=9, fontweight="semibold")

viz.label_axes(
    ax,
    title="Target-species occurrence records by year",
    subtitle="Records collapse after 2014 — the label supply, not the environmental data, bounds the training window.",
    xlabel="year", ylabel="records",
)
plt.tight_layout()
plt.show()

captured = yearly[(yearly.index >= START.year) & (yearly.index <= END.year)].sum()
print(f"records inside the window : {captured} ({captured / yearly.sum():.0%} of everything since 1990)")
print(f"records after {END.year}        : {yearly[yearly.index > END.year].sum()}")

That is the justification for the window. Extending it forward would add
environmental fields with essentially no labels attached; extending it backward
runs out of reliable reanalysis.

## 7. Raw zone summary

In [ ]:
rows = []
for path in sorted(config.RAW_DIR.rglob("*")):
    if path.is_file() and not path.name.startswith("."):
        rows.append({
            "file": str(path.relative_to(config.RAW_DIR)),
            "size_mb": round(path.stat().st_size / 1e6, 1),
        })
inventory = pd.DataFrame(rows).sort_values("size_mb", ascending=False)
print(f"total raw zone: {inventory.size_mb.sum():.0f} MB across {len(inventory)} files\n")
inventory

---

## What we have, and what is next

The raw zone now holds environmental fields on three different native grids,
plus point records from OBIS. Nothing has been aligned, cleaned, or joined yet.

**Notebook `01_eda`** examines what is actually in this data before any
modelling: the environmental niche of each species, how badly the occurrence
records are spatially biased, which covariates are redundant, and whether the
domain premise (fish aggregate at fronts and eddies) is visible at all.

A note on reproducibility: Copernicus periodically reprocesses these products,
so the dataset ID alone is not enough to reproduce a run. `fetch_physics` and
`fetch_bgc` write the dataset ID, version, bbox and date range into each cached
NetCDF's attributes.